<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/hallucination_filter_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title requirements
!pip install sentence-transformers jsonlines rouge_score transformers mistralai -q

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
#@title Confirm that the shared results directory exists
#@markdown Verify the directory path on your Google Drive, and change it if needed
import os
shared_path = "/content/drive/MyDrive/ColabNotebooks/SigExt/experiments" #@param{type:"string"}
print(os.path.exists(shared_path))

True


In [4]:
#@title Check if the required directories are present
base_path = shared_path

# check all needed files exist
files_to_check = [
    f"{base_path}/cnn_dataset_with_keyphrase/test.jsonl",
    f"{base_path}/cnn_extsig_predictions/test_predictions.json",
    f"{base_path}/cnn_extsig_predictions/test_dataset.jsonl",
    f"{base_path}/cnn_extsig_predictions/test_metrics.json",
]

for f in files_to_check:
  exists = os.path.exists(f)
  print(f"{'✓' if exists else '✗'}  {f}")

✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_with_keyphrase/test.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions/test_predictions.json
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions/test_dataset.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions/test_metrics.json


#Clone the repo

In [5]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"

# Preparing Data & Pipelines

In [6]:
# @title Download requirements for data preparing

%cd {shared_path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

/content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt/experiments


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [45]:
import os
import json
import nltk #imports the Natural Language Toolkit used to split text into sentences
import torch
import numpy as np
import tqdm #progress bar library
import jsonlines
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util #util provides the cosine similarity
from transformers import pipeline as hf_pipeline #provides AI models

#define all paths
base = shared_path
path_scored_dataset   = f"{base}/cnn_dataset_with_keyphrase/test.jsonl" #Path to the test articles that have phrase scores added by the Longformer
path_predictions      = f"{base}/cnn_extsig_predictions/test_predictions.json" #Path to the 500 raw summaries generated by Mistral
path_full_dataset     = f"{base}/cnn_extsig_predictions/test_dataset.jsonl" #Path to the full dataset file that includes raw_output
path_baseline_metrics = f"{base}/cnn_extsig_predictions/test_metrics.json" #Path to the ROUGE scores that were computed by the original SigExt pipeline
path_output           = f"{base}/hallucination_extension/cnn_verified_predictions/" #Path to the folder where the extension will save its results

os.makedirs(path_output, exist_ok=True)

TOP_N_EVIDENCE     = 5 #Controls how many source sentences are retrieved as evidence for each summary sentence
ENTAIL_THRESHOLD   = 0.3 #The minimum confidence score the NLI model must assign to "entailment" for a sentence to pass
MAX_REGEN_ATTEMPTS = 2 #How many times the extension tries to regenerate a flagged sentence before giving up and dropping it
SIMILARITY_THRESHOLD = 0.75

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cpu


In [48]:
#@title Load Models
print("Loading sentence embedder...")
embedder = SentenceTransformer("all-MiniLM-L6-v2") #Downloads and loads the sentence embedding model, used to produce meaningful sentence embeddings
embedder = embedder.to(device)

print("Loading NLI model...")
#Instead of manually tokenizing input, running the model, and decoding output, the pipeline handles everything automatically
nli_model = hf_pipeline(
    "text-classification",
    model="dleemiller/ModernCE-large-nli",
    device=0 if device == "cuda" else -1,
    top_k=None,
    batch_size=16,
    truncation=True,
    max_length=512,
)


Loading sentence embedder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading NLI model...


Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

In [9]:
import sys
import os
from types import ModuleType
from mistralai.client import Mistral
import tqdm

MISTRAL_API_KEY = "YdbLhqEepgXav0x8ruTJiWR20GsgWCrD"


def predict_one_eg_mistral(example): #Translte phrases for mistral
    client = Mistral(api_key=MISTRAL_API_KEY)
    try:
        chat_response = client.chat.complete(
            model="mistral-small-latest",
            messages=[{"role": "user", "content": example["prompt_input"]}]
        )
        return chat_response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

In [10]:
#@title Load Data

#raw summaries generated by the LLMs
with open(path_predictions) as f:
  raw_summaries = json.load(f)

#source articles with phrase scores
with jsonlines.open(path_scored_dataset) as f:
  scored_dataset = list(f)

#full dataset with raw_output for the ROUGE evaluation
with jsonlines.open(path_full_dataset) as f:
  full_dataset = list(f)

#baseline metrics for comparison
with open(path_baseline_metrics) as f:
  baseline_metrics = json.load(f)

print(f"Summaries loaded:       {len(raw_summaries)}")
print(f"Scored articles loaded: {len(scored_dataset)}")
print(f"Full dataset loaded:    {len(full_dataset)}")
print(f"\nBaseline ROUGE-1 F1: {baseline_metrics.get('rouge1f', 'N/A')}")
print(f"\nSample raw summary:\n{raw_summaries[0]}")

Summaries loaded:       500
Scored articles loaded: 500
Full dataset loaded:    500

Baseline ROUGE-1 F1: 4.7773

Sample raw summary:
A former Japanese school principal, Yuhei Takashima, is facing criminal charges after police in Tokyo arrested him for allegedly paying for sex with over 12,000 women—some as young as 14—in the Philippines, where he reportedly took 147,600 photos of his activities. Takashima admitted to police that he traveled to Manila repeatedly to "buy sex," calling it cheap, and his case, under investigation since 2013, involves allegations of obscene acts with minors.


#Verification Functions

In [32]:
'''
#sentence segmentation
def split_sentences(text):
  """
    Splits a given block of text into a list of individual sentences.

    Args:
        text (str): The input text block to be split.

    Returns:
        list[str]: A list of individual sentences, with leading and trailing
                   whitespace removed. Empty strings are excluded.

    Example:
        >>> split_sentences("Hello world! How are you? ")
        ['Hello world!', 'How are you?']
    """
  return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]


# evidence retrieval
def retrieve_evidence(summary_sentence, source_sentences, top_n=TOP_N_EVIDENCE):
  """
    Retrieves the most semantically similar evidence from a source article.

    Args:
        summary_sentence (str): One sentence from the generated summary.
        source_sentences (list[str]): The list of sentences from the source article.
        top_n (int, optional): The maximum number of evidence sentences to return.
                               Defaults to TOP_N_EVIDENCE (3).

    Returns:
        list[str]: A list containing the top `top_n` matching source sentences.
    """
  query_emb  = embedder.encode(summary_sentence, convert_to_tensor=True) #we need a tensor for the cos_sim
  source_emb = embedder.encode(source_sentences, convert_to_tensor=True)
  scores     = util.cos_sim(query_emb, source_emb) #cosine similarity between the  the summary sentence vector and every source sentence vector
  top_idx    = torch.topk(scores, k=min(top_n, len(source_sentences))).indices.flatten()
  return [source_sentences[idx.item()] for idx in top_idx]




#NLI entailment check
def check_entailment(summary_sentence, evidence):
  """
    Evaluates whether the source evidence supports the generated summary sentence.

    This function uses a Natural Language Inference (NLI) model to compare the
    evidence (premise) against the summary (hypothesis). It classifies the
    relationship as 'entailed', 'contradiction', or 'neutral'.

    Args:
        summary_sentence (str): The generated sentence being fact-checked (the hypothesis).
        evidence (list[str]): The list of source sentences acting as proof (the premise).

    Returns:
        tuple[str, float]: A tuple containing the final classification label
                           ("entailed", "contradiction", or "neutral") and its
                           corresponding confidence score.
    """
  premise = " ".join(evidence)
  result  = nli_model({"text": premise, "text_pair": summary_sentence}) #text is the premise, while text_pair is the hypothesis

  label_scores = {item["label"]: item["score"] for item in result}

  entail_score = label_scores.get("entailment", 0.0)
  if entail_score >= ENTAIL_THRESHOLD:
    return "entailed", entail_score
  elif label_scores.get("contradiction", 0) > label_scores.get("neutral",0):
    return "contradiction", label_scores.get("contradiction",0.0)
  else:
    return "neutral", label_scores.get("neutral",0.0)


#regenerate sentence (to be modified in the future
def regenerate_sentence(original_sentence: str, evidence: list[str]) -> str | None:
    """
    Prompts an LLM to rewrite an unsupported sentence using only verified evidence.

    If a generated sentence fails the NLI entailment check, this function attempts
    to salvage it by forcing a generation model to rewrite it strictly based on
    the provided source evidence.

    Args:
        original_sentence (str): The hallucinated or unsupported summary sentence.
        evidence (list[str]): The source sentences to use as ground-truth context.

    Returns:
        str | None: The rewritten, factually correct sentence. Returns None if
                    the model cannot rewrite it or explicitly outputs "UNSUPPORTED".
    """
    evidence_text = " ".join(evidence)
    prompt = (
        f"The following sentence may contain information not supported "
        f"by the source document.\n\n"
        f"Evidence from the source:\n{evidence_text}\n\n"
        f"Original sentence:\n{original_sentence}\n\n"
        f"Rewrite the sentence using only information from the evidence. "
        f"Do not add anything not in the evidence. "
        f'If impossible, respond with exactly "UNSUPPORTED".'
    )
    result = predict_one_eg_mistral({"prompt_input": prompt})
    if not result or result.strip().upper() == "UNSUPPORTED":
        return None
    return result.strip()



# ── Stage 8 — Pipeline Orchestrator (Verification Loop) ──────────────────

# Number of times the LLM will attempt to rewrite a hallucinated sentence
MAX_REGEN_ATTEMPTS = 3

def verify_summary(raw_summary: str, source_document: str) -> tuple[str, list[dict]]:
    """
    Orchestrates the fact-checking pipeline to verify and correct a generated summary.

    This function splits the summary and source document into sentences, checks
    each summary sentence for entailment against the source, and attempts to
    regenerate any unsupported sentences. It strips out sentences that cannot
    be verified or corrected.

    Args:
        raw_summary (str): The initial, unverified summary text generated by the model.
        source_document (str): The original source article used as ground truth.

    Returns:
        tuple[str, list[dict]]: A tuple containing:
            - final_summary (str): The cleaned, verified summary text.
            - halluc_log (list[dict]): A detailed log of all failed sentences,
              their NLI scores, and regeneration attempts for analytics.
    """
    # 1. Preprocessing: Break text blocks down into iterable sentences
    summary_sentences = split_sentences(raw_summary)
    source_sentences  = split_sentences(source_document)

    # 2. Pipeline Safety: Edge case where either input is completely empty
    if not summary_sentences or not source_sentences:
        return raw_summary, []

    verified = []    # Stores sentences that pass verification
    halluc_log = []  # Stores audit trails for sentences that fail

    # 3. Main processing loop: Evaluate one summary sentence at a time
    for i, sentence in enumerate(summary_sentences):

        # Step A: Find the most relevant facts in the source document
        evidence = retrieve_evidence(sentence, source_sentences)

        # Step B: Check if the evidence actually supports the sentence
        label, score = check_entailment(sentence, evidence)

        # Step C: If it passes, save it and move to the next sentence
        if label == "entailed":
            verified.append(sentence)
            continue

        # Step D: If it fails, create an audit log entry for analytics
        log_entry = {
            "sentence_idx":   i,
            "original":       sentence,
            "label":          label,
            "score":          round(score, 4),
            "evidence":       evidence,
            "regenerated":    None,
            "regen_accepted": False,
        }

        # Step E: Attempt to fix the hallucinated sentence (Retry Loop)
        regenerated = None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            # Ask the LLM to rewrite it
            candidate = regenerate_sentence(sentence, evidence)

            # If the LLM gives up (returns None), stop trying
            if candidate is None:
                break

            # Verify the newly rewritten sentence using the same NLI check
            new_label, new_score = check_entailment(candidate, evidence)
            if new_label == "entailed":
                regenerated = candidate
                log_entry["regen_accepted"] = True
                break # Success! Exit the retry loop early

        # Step F: Finalize the audit log for this sentence
        log_entry["regenerated"] = regenerated
        halluc_log.append(log_entry)

        # Step G: If we successfully salvaged the sentence, add it to the final output
        if regenerated:
            verified.append(regenerated)

    # 4. Post-processing: Reconstruct the final, clean summary
    final_summary = " ".join(verified)

    return final_summary, halluc_log

print("Verification functions defined.")
'''

Verification functions defined.


In [47]:
#@title Verification Functions

import re
import nltk
import torch
from sentence_transformers import util

# ── Stage 4 — sentence segmentation ──────────────────────────
def split_sentences(text):
    if not text or not text.strip():
        return []
    return [
        s.strip()
        for s in nltk.sent_tokenize(text)
        if s.strip() and len(s.split()) >= 3
    ]


# ── Stage 5 — evidence retrieval ─────────────────────────────
def retrieve_evidence(summary_sentence, source_sentences, top_n=TOP_N_EVIDENCE):
    """
    Retrieves the most semantically similar source sentences as evidence.
    Uses cosine similarity between sentence embeddings.
    Returns top_n most similar source sentences.
    """
    query_emb  = embedder.encode(summary_sentence, convert_to_tensor=True)
    source_emb = embedder.encode(source_sentences, convert_to_tensor=True)
    scores     = util.cos_sim(query_emb, source_emb)
    top_idx    = torch.topk(
        scores, k=min(top_n, len(source_sentences))
    ).indices.flatten()
    return [source_sentences[idx.item()] for idx in top_idx]


# ── Stage 6 — hybrid entailment check ────────────────────────
def check_entailment(summary_sentence, evidence):
    """
    Hybrid check combining semantic similarity and NLI.

    A sentence passes if:
    1. NLI entailment score >= threshold for ANY evidence sentence
    OR
    2. Semantic similarity >= similarity threshold for ANY evidence sentence
       (catches paraphrases that NLI misses)

    A sentence is only flagged as hallucination if:
    - Both NLI and similarity fail
    - AND NLI returns contradiction (not just neutral)
    """
    best_entail  = 0.0
    best_contra  = 0.0
    best_neutral = 0.0
    best_sim     = 0.0

    # encode summary sentence once
    summary_emb = embedder.encode(summary_sentence, convert_to_tensor=True)

    for ev_sentence in evidence:

        # compute semantic similarity
        ev_emb = embedder.encode(ev_sentence, convert_to_tensor=True)
        sim    = float(util.cos_sim(summary_emb, ev_emb)[0][0])
        if sim > best_sim:
            best_sim = sim

        # compute NLI
        result = nli_model({
            "text":      ev_sentence,
            "text_pair": summary_sentence
        })
        label_scores = {item["label"]: item["score"] for item in result}

        entail  = label_scores.get("entailment",    0.0)
        contra  = label_scores.get("contradiction", 0.0)
        neutral = label_scores.get("neutral",       0.0)

        if entail  > best_entail:  best_entail  = entail
        if contra  > best_contra:  best_contra  = contra
        if neutral > best_neutral: best_neutral = neutral

    # pass conditions:
    # 1. NLI says entailed
    # 2. OR sentences are very semantically similar (paraphrase)
    if best_entail >= ENTAIL_THRESHOLD or best_sim >= SIMILARITY_THRESHOLD:
        return "entailed", max(best_entail, best_sim)

    # fail conditions — only flag real hallucinations
    elif best_contra >= 0.7:
        return "contradiction", best_contra
    else:
        # neutral with low similarity — invented information
        return "neutral", best_neutral

# ── Stage 7 — sentence regeneration ──────────────────────────
def regenerate_sentence(
    original_sentence,
    evidence,
    predict_fn=predict_one_eg_mistral
):
    """
    Asks the LLM to rewrite a flagged sentence using only the evidence.
    Returns None if the LLM cannot produce a faithful rewrite.
    Keyphrases are NOT passed here to avoid reproducing the same error.
    """
    evidence_text = " ".join(evidence)
    prompt = (
        f"The following sentence may contain information not supported "
        f"by the source document.\n\n"
        f"Evidence from the source:\n{evidence_text}\n\n"
        f"Original sentence:\n{original_sentence}\n\n"
        f"Rewrite the sentence using only information from the evidence. "
        f"Do not add anything not in the evidence. "
        f'If impossible, respond with exactly "UNSUPPORTED".'
    )
    result = predict_fn({"prompt_input": prompt})
    if not result or result.strip().upper() == "UNSUPPORTED":
        return None
    return result.strip()


# ── Stages 4-8 — pipeline orchestrator ───────────────────────
def verify_summary(
    raw_summary,
    source_document,
    predict_fn=predict_one_eg_mistral
):
    """
    Orchestrates the full verification pipeline for one summary.

    For each summary sentence:
        1. Retrieve top-N most similar source sentences as evidence
        2. Check pairwise NLI entailment against each evidence sentence
        3. If passes → keep sentence unchanged
        4. If fails  → attempt regeneration using evidence as constraint
        5. If regeneration passes NLI → use regenerated sentence
        6. If regeneration fails     → drop sentence

    Returns the verified summary and a log of all hallucinations found.
    """
    summary_sentences = split_sentences(raw_summary)
    source_sentences  = split_sentences(source_document)

    if not summary_sentences or not source_sentences:
        return raw_summary, []

    verified   = []
    halluc_log = []

    for i, sentence in enumerate(summary_sentences):

        # Stage 5 — retrieve evidence
        evidence = retrieve_evidence(sentence, source_sentences)

        # Stage 6 — pairwise entailment check
        label, score = check_entailment(sentence, evidence)

        # sentence passes — keep it unchanged
        if label == "entailed":
            verified.append(sentence)
            continue

        # sentence flagged — create audit log entry
        log_entry = {
            "sentence_idx":   i,
            "original":       sentence,
            "label":          label,
            "score":          round(score, 4),
            "evidence":       evidence,
            "regenerated":    None,
            "regen_accepted": False,
        }

        # Stage 7 — attempt regeneration
        regenerated = None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            candidate = regenerate_sentence(sentence, evidence, predict_fn)

            if candidate is None:
                break

            # verify the regenerated sentence with pairwise NLI
            new_label, new_score = check_entailment(candidate, evidence)
            if new_label == "entailed":
                regenerated = candidate
                log_entry["regen_accepted"] = True
                break

        # Stage 8 — finalize log and add to verified if successful
        log_entry["regenerated"] = regenerated
        halluc_log.append(log_entry)

        if regenerated:
            verified.append(regenerated)

    # reassemble final summary
    final_summary = " ".join(verified)
    return final_summary, halluc_log


print("Verification functions defined.")

Verification functions defined.


#Final Pipeline

In [49]:
final_summaries = []
all_halluc_logs  = []


"""
At each iteration:
i = 0
raw_summary = "Scientists confirmed the Higgs boson. The cost was 13 billion."
article     = {
    "trunc_input": "Scientists at CERN confirmed the Higgs boson particle.
                    The research involved 6,000 scientists...",
    "input_kw_model": [...],
    ...
}
"""
for i, (raw_summary, article) in enumerate(
    tqdm.tqdm(
        zip(raw_summaries, scored_dataset),
        total=len(raw_summaries),
        desc="Verifying"
    )
):
    # skip empty summaries
    if not raw_summary or not raw_summary.strip():
        final_summaries.append("")
        all_halluc_logs.append({
            "example_idx":    i,
            "raw_summary":    raw_summary,
            "final_summary":  "",
            "hallucinations": [],
        })
        continue

    source_document = article["trunc_input"]
    final_summary, halluc_log = verify_summary(raw_summary, source_document)

    final_summaries.append(final_summary)
    all_halluc_logs.append({
        "example_idx":    i,
        "raw_summary":    raw_summary,
        "final_summary":  final_summary,
        "hallucinations": halluc_log,
    })

print(f"\nVerification complete.")
print(f"Total examples processed: {len(final_summaries)}")
print(f"Total hallucinations found: {sum(len(l['hallucinations']) for l in all_halluc_logs)}")

Verifying: 100%|██████████| 500/500 [2:04:29<00:00, 14.94s/it]


Verification complete.
Total examples processed: 500
Total hallucinations found: 33


In [50]:
#@title Evaluate and save results (Stage 9)
from rouge_score import rouge_scorer
import numpy as np
import json
import os

def compute_rouge(predictions, references):
    scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"], use_stemmer=True
    )
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)
    return {
        "rouge1f": round(np.mean(r1_f) * 100, 4),
        "rougeLf": round(np.mean(rl_f) * 100, 4),
        "rouge1r": round(np.mean(r1_r) * 100, 4),
    }

# human reference summaries
references = [item["raw_output"] for item in full_dataset]

# compute ROUGE for raw and verified summaries
raw_metrics   = compute_rouge(raw_summaries,   references)
final_metrics = compute_rouge(final_summaries, references)

# hallucination statistics
total_sentences = sum(
    len(split_sentences(log["raw_summary"]))
    for log in all_halluc_logs
    if log["raw_summary"]
)
total_flagged = sum(
    len(log["hallucinations"])
    for log in all_halluc_logs
)
regen_accepted = sum(
    sum(1 for h in log["hallucinations"] if h["regen_accepted"])
    for log in all_halluc_logs
)
dropped      = total_flagged - regen_accepted
halluc_rate  = total_flagged / total_sentences if total_sentences > 0 else 0

# print results table
print("=" * 52)
print(f"{'Metric':<28} {'Baseline':>10} {'Verified':>10}")
print("=" * 52)
print(f"{'ROUGE-1 F1':<28} {raw_metrics['rouge1f']:>10} {final_metrics['rouge1f']:>10}")
print(f"{'ROUGE-L F1':<28} {raw_metrics['rougeLf']:>10} {final_metrics['rougeLf']:>10}")
print(f"{'ROUGE-1 Recall':<28} {raw_metrics['rouge1r']:>10} {final_metrics['rouge1r']:>10}")
print("=" * 52)
print(f"{'Total sentences':<28} {total_sentences:>21}")
print(f"{'Flagged sentences':<28} {total_flagged:>21}")
print(f"{'Hallucination rate':<28} {halluc_rate:>20.2%}")
print(f"{'Regenerated accepted':<28} {regen_accepted:>21}")
print(f"{'Dropped':<28} {dropped:>21}")
print("=" * 52)

# save results to Drive
results = {
    "raw_metrics":        raw_metrics,
    "verified_metrics":   final_metrics,
    "hallucination_rate": round(halluc_rate, 4),
    "total_sentences":    total_sentences,
    "total_flagged":      total_flagged,
    "regen_accepted":     regen_accepted,
    "dropped":            dropped,
}

os.makedirs(path_output, exist_ok=True)

with open(f"{path_output}metrics_verified.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics_verified.json saved")

with open(f"{path_output}halluc_log.json", "w") as f:
    json.dump(all_halluc_logs, f, indent=2)
print("halluc_log.json saved")

with open(f"{path_output}final_predictions.json", "w") as f:
    json.dump(final_summaries, f, indent=2)
print("final_predictions.json saved")

# verify files exist and show sizes
print("\nFiles saved:")
for fname in ["metrics_verified.json", "halluc_log.json", "final_predictions.json"]:
    fpath = f"{path_output}{fname}"
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  {fname}  ({size:,} bytes)")

Metric                         Baseline   Verified
ROUGE-1 F1                       4.7773     4.5603
ROUGE-L F1                       3.0558     2.9816
ROUGE-1 Recall                   5.8398      5.359
Total sentences                                999
Flagged sentences                               33
Hallucination rate                          3.30%
Regenerated accepted                            21
Dropped                                         12

metrics_verified.json saved
halluc_log.json saved
final_predictions.json saved

Files saved:
  metrics_verified.json  (311 bytes)
  halluc_log.json  (309,173 bytes)
  final_predictions.json  (107,660 bytes)


In [13]:
#@title Evaluate and save results
from rouge_score import rouge_scorer
import numpy as np
import json
import os

def compute_rouge(predictions, references):
    scorer_obj = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"], use_stemmer=True
    )
    r1_f, rl_f, r1_r = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not pred.strip():
            pred = "empty"
        score = scorer_obj.score(target=ref, prediction=pred)
        r1_f.append(score["rouge1"].fmeasure)
        rl_f.append(score["rougeL"].fmeasure)
        r1_r.append(score["rouge1"].recall)
    return {
        "rouge1f": round(np.mean(r1_f) * 100, 4),
        "rougeLf": round(np.mean(rl_f) * 100, 4),
        "rouge1r": round(np.mean(r1_r) * 100, 4),
    }

# human reference summaries
references = [item["raw_output"] for item in full_dataset]

# compute ROUGE for raw and verified summaries
raw_metrics   = compute_rouge(raw_summaries,   references)
final_metrics = compute_rouge(final_summaries, references)

# hallucination statistics
total_sentences = sum(
    len(split_sentences(log["raw_summary"]))
    for log in all_halluc_logs
    if log["raw_summary"]
)
total_flagged = sum(
    len(log["hallucinations"])
    for log in all_halluc_logs
)
regen_accepted = sum(
    sum(1 for h in log["hallucinations"] if h["regen_accepted"])
    for log in all_halluc_logs
)
dropped      = total_flagged - regen_accepted
halluc_rate  = total_flagged / total_sentences if total_sentences > 0 else 0

# print results table
print("=" * 52)
print(f"{'Metric':<28} {'Baseline':>10} {'Verified':>10}")
print("=" * 52)
print(f"{'ROUGE-1 F1':<28} {raw_metrics['rouge1f']:>10} {final_metrics['rouge1f']:>10}")
print(f"{'ROUGE-L F1':<28} {raw_metrics['rougeLf']:>10} {final_metrics['rougeLf']:>10}")
print(f"{'ROUGE-1 Recall':<28} {raw_metrics['rouge1r']:>10} {final_metrics['rouge1r']:>10}")
print("=" * 52)
print(f"{'Total sentences':<28} {total_sentences:>21}")
print(f"{'Flagged sentences':<28} {total_flagged:>21}")
print(f"{'Hallucination rate':<28} {halluc_rate:>20.2%}")
print(f"{'Regenerated accepted':<28} {regen_accepted:>21}")
print(f"{'Dropped':<28} {dropped:>21}")
print("=" * 52)

# save results to Drive
results = {
    "raw_metrics":        raw_metrics,
    "verified_metrics":   final_metrics,
    "hallucination_rate": round(halluc_rate, 4),
    "total_sentences":    total_sentences,
    "total_flagged":      total_flagged,
    "regen_accepted":     regen_accepted,
    "dropped":            dropped,
}

os.makedirs(path_output, exist_ok=True)

with open(f"{path_output}metrics_verified.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nmetrics_verified.json saved")

with open(f"{path_output}halluc_log.json", "w") as f:
    json.dump(all_halluc_logs, f, indent=2)
print("halluc_log.json saved")

with open(f"{path_output}final_predictions.json", "w") as f:
    json.dump(final_summaries, f, indent=2)
print("final_predictions.json saved")

# verify files exist and show sizes
print("\nFiles saved:")
for fname in ["metrics_verified.json", "halluc_log.json", "final_predictions.json"]:
    fpath = f"{path_output}{fname}"
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  {fname}  ({size:,} bytes)")

Metric                         Baseline   Verified
ROUGE-1 F1                       4.7773     15.284
ROUGE-L F1                       3.0558    10.6508
ROUGE-1 Recall                   5.8398    13.9873
Total sentences                                999
Flagged sentences                              954
Hallucination rate                         95.50%
Regenerated accepted                           405
Dropped                                        549

metrics_verified.json saved
halluc_log.json saved
final_predictions.json saved

Files saved:
  metrics_verified.json  (317 bytes)
  halluc_log.json  (956,786 bytes)
  final_predictions.json  (70,821 bytes)
